In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_23_renormalization_group_and_finite_size_scaling.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 23: Phase Transitions, Cluster Algorithms, Renormalization Group & Finite-Size Scaling

_Session 17749b2b · Estimated time: ~115 minutes · 42 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Quantify critical slowing down near second-order phase transitions using the correlation length divergence $\xi \propto |t|^{-\nu}$, autocorrelation time $\tau \propto \xi^z \sim L^z$, and total CPU effort $\propto L^{d+z}$.
- Derive the Fortuin–Kasteleyn bond activation probability $p = 1 - e^{-2\beta J}$ for Swendsen–Wang and Wolff cluster updates, proving how it identically cancels the boundary surface-energy Boltzmann factor $e^{-2\beta m J}$ to yield rejection-free cluster flips ($A = 1$).
- Define all six static critical exponents ($\alpha, \beta, \gamma, \delta, \nu, \eta$), compare their values across Mean-Field, 2D Ising, and 3D Ising universality classes, and apply the Rushbrooke, Widom, Fisher, and Josephson hyperscaling identities.
- Formulate Kadanoff block-spin transformations ($L' = L/b$) and Wilson's Renormalization Group (RG) flow $K' = R(K)$, distinguishing stable trivial sinks ($K=0, \infty$) from the unstable critical fixed point $K^*$ and extracting $\nu = \ln b / \ln(dR/dK|_{K^*})$.
- Construct the Finite-Size Scaling (FSS) ansatz from generalized homogeneous free energies, perform multi-lattice data collapse ($M L^{\beta/\nu}$ and $\chi L^{-\gamma/\nu}$ vs. $t L^{1/\nu}$), and locate $T_c$ without prior exponent knowledge via Binder's 4th-order cumulant crossing $U_4(T, L) = 1 - \frac{\langle M^4 \rangle}{3\langle M^2 \rangle^2}$.


In [2]:
ws.render_section('sec_ising_critical_slowing_down')

### The Ising Model and Phase Diagram
Consider an $L^d$ lattice of spins $s_i = \pm 1$ interacting via nearest-neighbor coupling $J$ in an external magnetic field $h$:
$$H = -J \sum_{\langle ij \rangle} s_i s_j - h \sum_{i=1}^N s_i, \qquad Z = \sum_{\{s_i = \pm 1\}} e^{-\beta H}$$
- For **$J > 0$ (ferromagnetic)** at $h = 0$, the system transitions from a high-temperature disordered **paramagnet** ($\langle M \rangle = 0$ for $T > T_c$) to a low-temperature ordered **ferromagnet** ($\langle M \rangle = \pm M_0 \neq 0$ for $T < T_c$).
- Sweeping the magnetic field $h$ across $h = 0$ at fixed $T < T_c$ triggers a **first-order phase transition** (a discontinuous jump in magnetization from $-M_0$ to $+M_0$ accompanied by **hysteresis**, metastability, and surface-tension nucleation barriers, directly analogous to crossing the liquid–vapor coexistence curve in pressure–temperature space).
- Approaching $(T = T_c, h = 0)$ along $h = 0$ produces a **continuous (second-order) phase transition** at the critical point, where droplets/domains of correlated spins fluctuate across **all length scales** (manifest experimentally as **critical opalescence** in fluids like $\text{CO}_2$, where density fluctuations reach the wavelength of visible light).

---

### Critical Slowing Down and the Dynamical Exponent $z$
As temperature approaches $T_c$ in terms of the **reduced temperature** $t \equiv \frac{T - T_c}{T_c}$, the spatial **correlation length** $\xi(t)$ diverges as a power law in the thermodynamic limit:
$$\xi(t) \propto |t|^{-\nu}$$
In a finite simulation box of linear dimension $L$, $\xi$ cannot exceed the box size and is cut off at $\xi \sim L$ near $T_c$.

When using any **local single-spin update** (such as standard Metropolis or Glauber dynamics), information diffuses across correlated domains via a random walk of domain walls ($\tau_0 \propto \xi^2 / D$). Consequently, the **autocorrelation time** $\tau$ (measured in Monte Carlo sweeps per spin) diverges with $\xi$ according to the **critical dynamical exponent $z$**:
$$\tau \propto \xi^z \;\;\xrightarrow{\text{at } T \approx T_c}\;\; \tau(L) \propto L^z$$
Because each Monte Carlo sweep over the lattice requires $O(L^d)$ single-spin operations and one must run for $\propto \tau(L) \propto L^z$ sweeps to generate a single statistically independent configuration, the **total CPU time per independent sample** at $T_c$ scales as:
$$\text{CPU Effort per Independent Sample} \propto L^d \cdot \tau(L) \propto L^{d + z}$$
- **Local Metropolis / Glauber updates:** $z \approx 2.125$ in 2D ($d+z \approx 4.125$) and $z \approx 2.02$ in 3D ($d+z \approx 5.02$). Doubling $L$ in 3D increases the CPU cost by $2^5 = 32\times$!
- **Low-$T$ and High-$T$ Extremes:** At very low $T \ll T_c$, local flips are almost never accepted ($\Delta E > 0$) and a huge free-energy barrier $\Delta F \propto \sigma L^{d-1}$ separates the $+M_0$ and $-M_0$ basins (**quasi-ergodicity**).

---

### Non-Local Cluster Algorithms: Swendsen–Wang (1987) & Wolff (1989)
To defeat critical slowing down, **Swendsen and Wang (PRL 58, 86, 1987)** and **Wolff (PRL 62, 361, 1989)** introduced non-local cluster updates based on the **Fortuin–Kasteleyn (FK) bond representation**:
1. Examine every nearest-neighbor pair $\langle ij \rangle$. If the spins are **anti-aligned** ($s_i \neq s_j$), place **no bond** ($n_{ij} = 0$).
2. If the spins are **aligned** ($s_i = s_j$), activate a bond ($n_{ij} = 1$) with probability:
 $$p = 1 - e^{-2\beta J} = 1 - \exp\!\left(-\frac{2J}{k_B T}\right)$$
3. Identify connected clusters of bonded spins:
 - **Swendsen–Wang:** Partition the entire lattice into connected clusters (using the Hoshen–Kopelman algorithm) and flip *each* cluster independently with probability $1/2$ (or assign random $q$-state Potts values).
 - **Wolff single-cluster:** Pick a single random seed spin, recursively grow its connected cluster by activating bonds to aligned neighbors with probability $p = 1 - e^{-2\beta J}$, and flip that entire cluster with probability $1$.

#### Proof of Rejection-Free Detailed Balance ($A = 1$)
Suppose a constructed cluster $C$ has $m_+$ aligned pairs ($s_i = s_j$) along its perimeter where an FK bond was **not** placed (each occurred with probability $1 - p$), and $m_-$ anti-aligned pairs ($s_i \neq s_j$) along its perimeter. When the entire cluster $C$ is flipped ($C \to C'$):
- Internal bonds inside $C$ do not change energy because both spins flip together.
- Along the boundary of $C$, the $m_+$ previously aligned perimeter pairs become anti-aligned, and the $m_-$ previously anti-aligned perimeter pairs become aligned! Thus the energy change is:
 $$\Delta E = E(\sigma') - E(\sigma) = 2J(m_+ - m_-)$$
- To form the exact same cluster $C$ starting from the flipped state $\sigma'$, the algorithm must fail to place FK bonds across the $m_-$ newly aligned boundary edges (probability $(1-p)^{m_-}$), while all internal bond probabilities are identical:
 $$\frac{T(\sigma \to \sigma')}{T(\sigma' \to \sigma)} = \frac{(1 - p)^{m_+}}{(1 - p)^{m_-}} = (1 - p)^{m_+ - m_-}$$
- Enforcing **Detailed Balance** requires the Metropolis acceptance ratio $A(\sigma \to \sigma')$ to satisfy:
 $$\frac{A(\sigma \to \sigma')}{A(\sigma' \to \sigma)} = \frac{P_{\text{eq}}(\sigma')}{P_{\text{eq}}(\sigma)} \frac{T(\sigma' \to \sigma)}{T(\sigma \to \sigma')} = e^{-\beta \Delta E} (1 - p)^{-(m_+ - m_-)} = \left[e^{-2\beta J} (1 - p)^{-1}\right]^{m_+ - m_-}$$
- Choosing **$1 - p = e^{-2\beta J} \iff p = 1 - e^{-2\beta J}$** makes the bracketed factor equal **1** identically for any boundary geometry ($m_+, m_-$), permitting **$A(\sigma \to \sigma') = 1$ (100% rejection-free cluster flips)**!
- As a result, Swendsen–Wang slashes the dynamical exponent to **$z \approx 0.35$ in 2D** and **$z \approx 0.50$ in 3D**!


In [3]:
ws.render_section('concept_swendsen_wang_slowing_down')

### Interactive Explorer: Critical Slowing Down vs. Swendsen–Wang Cluster Scaling

Switch between the 2D and 3D Ising models, vary the box size $L$, and toggle between autocorrelation time $\tau(L) \propto L^z$ and total CPU effort $\propto L^{d+z}$.

**Key Takeaway:** Because local single-spin flips only move domain boundaries by diffusive random walks ($z \approx 2$), large critical droplets take $\tau \propto L^2$ sweeps to decorrelate. Fortuin–Kasteleyn cluster algorithms percolate directly across critical droplets of size $\xi \sim L$ and flip them rejection-free ($A=1$), collapsing $z$ to $0.35$ in 2D.

In [4]:
ws.render_section('q23_01')

**Q1 · easy** · 💡

In a Monte Carlo simulation of a $d$-dimensional system right at its critical temperature $T_c$ (where the bulk correlation length is cut off by the box size, $\xi \sim L$), how does the **total CPU time** required to generate one statistically independent configuration scale with linear system size $L$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the computational work needed to perform a single full lattice sweep alongside how the relaxation time (in units of sweeps) grows near criticality as a function of the system dimension and dynamical critical exponent.

</div>
</details>

- [ ] **(A)** $\propto L^z$, because each sweep requires $O(1)$ operations regardless of lattice volume.
- [ ] **(B)** $\propto L^d$, because the autocorrelation time $\tau$ approaches a size-independent constant at $T_c$.
- [ ] **(C)** $\propto L^{d+z}$, because each sweep over the lattice costs $O(L^d)$ operations and $\tau \propto L^z$ sweeps are needed per independent sample.
- [ ] **(D)** $\propto L^{d-z}$, because larger boxes average out spatial fluctuations faster.

In [5]:
ws.render_section('q23_02')

**Q2 · medium** · 💡

In a 2D Ising simulation at $T_c$, local Metropolis dynamics has dynamical exponent $z_{\text{local}} = 2.125$, whereas the Swendsen–Wang cluster algorithm has $z_{\text{SW}} = 0.35$. If you increase the linear lattice size from $L = 10$ to $L = 100$ (a factor of $10$), by what factor does the **ratio of autocorrelation times** $\tau_{\text{local}}(L) / \tau_{\text{SW}}(L)$ grow? *(Round to the nearest tenth.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Express each algorithm's autocorrelation scaling in terms of the linear lattice dimension, form the relative ratio across the two length scales, and evaluate the difference in dynamical critical exponents.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q23_03')

**Q3 · medium** · 💡

Using a local single-spin Metropolis algorithm in $d = 3$ dimensions with dynamical exponent $z = 2.0$, by what factor does the **total CPU time per independent sample** increase at $T_c$ when the linear box size $L$ is doubled ($L \to 2L$)? *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the total computational effort to both the volume cost of sweeping the degrees of freedom and the critical slowing down exponent upon scaling the linear dimension.

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q23_04')

**Q4 · medium** · 💡

Why does choosing the Fortuin–Kasteleyn bond activation probability $p = 1 - e^{-2\beta J}$ between aligned nearest-neighbor spins ($s_i = s_j$) guarantee a **rejection-free** ($A = 1$) cluster flip in the Swendsen–Wang and Wolff algorithms?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine how the conditional probabilities of satisfying or omitting bond connections along the perimeter of the cluster balance against the change in the system's Boltzmann weight under a collective spin inversion.

</div>
</details>

- [ ] **(A)** Because every cluster formed with $p = 1 - e^{-2\beta J}$ has exactly zero net boundary energy change ($\Delta E = 0$).
- [ ] **(B)** Because the probability $1 - p = e^{-2\beta J}$ of *not* placing a bond across each of the $m_+$ aligned perimeter edges identically cancels the Boltzmann surface-energy penalty $e^{-2\beta J (m_+ - m_-)}$ in the detailed balance ratio.
- [ ] **(C)** Because internal bonds inside the cluster flip their relative alignment and release enough energy to pay for the perimeter bonds.
- [ ] **(D)** Because the cluster algorithm only flips spins in the paramagnetic phase where $J = 0$.

In [8]:
ws.render_section('q23_05')

**Q5 · medium** · 💡

At the exact critical temperature of the 2D square-lattice Ising model, $\frac{k_B T_c}{J} = \frac{2}{\ln(1 + \sqrt{2})} \approx 2.2692$ (so $\beta_c J = \frac{1}{2}\ln(1 + \sqrt{2}) \approx 0.4407$). What is the exact Fortuin–Kasteleyn bond activation probability between aligned neighboring spins at $T_c$? *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the standard Fortuin–Kasteleyn bond weight for parallel nearest neighbors using the exact critical coupling $\beta_c J$.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('q23_06')

**Q6 · easy** · 💡

How do the **Swendsen–Wang** and **Wolff** cluster algorithms differ in how they update spin configurations during a single step?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider whether the entire lattice is partitioned into clusters or a single cluster is grown from a random seed spin.

</div>
</details>

- [ ] **(A)** Swendsen–Wang partitions the entire lattice into connected FK clusters (via Hoshen–Kopelman) and flips each cluster with probability $1/2$, whereas Wolff grows a single connected cluster from a randomly chosen seed spin and flips it with probability $1$.
- [ ] **(B)** Swendsen–Wang bonds anti-aligned spins ($s_i \neq s_j$), whereas Wolff bonds aligned spins ($s_i = s_j$).
- [ ] **(C)** Swendsen–Wang requires a Metropolis rejection step, whereas Wolff is rejection-free.
- [ ] **(D)** Wolff updates only single isolated spins, whereas Swendsen–Wang updates $2\times 2$ Kadanoff blocks.

In [10]:
ws.render_section('q23_07')

**Q7 · medium** · 💡

Which of the following statements accurately contrast a **first-order** phase transition (such as sweeping magnetic field $h$ across $0$ at $T < T_c$) with a **second-order (continuous)** critical transition ($T \to T_c$ at $h = 0$)? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Contrast the continuity of order parameters, latent heat, and divergent spatial fluctuation lengths against mixed-phase coexistence and free energy barriers.

</div>
</details>

- [ ] **(A)** At a second-order critical point, the correlation length $\xi$ diverges ($\xi \to \infty$ in the bulk) and fluctuations occur across all length scales.
- [ ] **(B)** At a first-order transition, the order parameter jumps discontinuously and simulations exhibit hysteresis, metastability, and interface/surface-energy barriers.
- [ ] **(C)** In a fluid like $\text{CO}_2$ near its critical point, critical opalescence arises because droplet/density correlation lengths grow comparable to the wavelength of visible light.
- [ ] **(D)** First-order transitions exhibit a diverging correlation length $\xi = \infty$ with zero interfacial free energy between coexisting phases.

In [11]:
ws.render_section('q23_08')

**Q8 · medium** · 💡

The statistical efficiency $\zeta$ of a Monte Carlo estimator for an observable $O$ is defined in terms of the squared statistical variance of the mean $v = \sigma_{\bar{O}}^2$ and total computational run time $T_{\text{CPU}}$ as $\zeta = (v \, T_{\text{CPU}})^{-1}$. Which expression is equivalent to $\zeta$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Express the variance of the sample mean in terms of the intrinsic variance of the observable, the integrated autocorrelation time, and the number of independent statistical blocks accumulated over the run.

</div>
</details>

- [ ] **(A)** $\zeta = \left[\text{var}(O)\\,\tau_0\\,(\text{CPU time per step})\right]^{-1}$
- [ ] **(B)** $\zeta = \text{var}(O)\\,\tau_0\\,(\text{CPU time per step})$
- [ ] **(C)** $\zeta = \tau_0^2 / \text{var}(O)$
- [ ] **(D)** $\zeta = L^z / \tau_0$

In [12]:
ws.render_section('sec_critical_exponents_universality')

### Definitions of the Six Static Critical Exponents
In the vicinity of a continuous critical point $T_c$, thermodynamic observables and correlation functions exhibit singular power-law behavior in the **reduced temperature** $t \equiv \frac{T - T_c}{T_c}$ and external field $h$:

1. **Order Parameter (Magnetization $M$ or density difference $\rho_L - \rho_G$):**
 $$M(t, h=0) \propto (-t)^\beta \qquad (T < T_c)$$
2. **Zero-Field Susceptibility ($\chi = \left.\frac{\partial M}{\partial h}\right|_{h=0}$) / Compressibility:**
 $$\chi(t, h=0) \propto |t|^{-\gamma}$$
3. **Zero-Field Specific Heat ($C_V = -T \frac{\partial^2 F}{\partial T^2}$):**
 $$C_V(t, h=0) \propto |t|^{-\alpha}$$
4. **Critical Isotherm ($T = T_c$, i.e. $t = 0$):**
 $$M(t=0, h) \propto |h|^{1/\delta} \;\implies\; h \propto |M|^\delta$$
5. **Correlation Length ($\xi$, characterizing spatial decay $G(r) \sim e^{-r/\xi}$ off criticality):**
 $$\xi(t, h=0) \propto |t|^{-\nu}$$
6. **Anomalous Dimension ($\eta$, governing power-law decay of spin–spin correlation at $T = T_c$):**
 $$G(r) = \langle s_0 s_r \rangle - \langle s \rangle^2 \propto \frac{1}{r^{d - 2 + \eta}} \qquad (\text{at } T = T_c, \; \xi = \infty)$$

---

### Comparison of Critical Exponents Across Universality Classes
From Table 3.1 of the lecture slides:

| Exponent | Physical Quantity | Mean Field (Landau, $d \ge 4$) | 2D Ising ($d=2, n=1$) | 3D Ising ($d=3, n=1$) | Experiment (Fluids / 3D Magnets) |
|:---: |:--- |:---: |:---: |:---: |:---: |
| **$\alpha$** | Specific heat $C_V \propto \|t\|^{-\alpha}$ | $0$ (discontinuity) | $0$ (logarithmic) | $0.110(5)$ | $0.110 - 0.116$ |
| **$\beta$** | Order parameter $M \propto (-t)^\beta$ | $1/2 = 0.5$ | $1/8 = 0.125$ | $0.325 \pm 0.0015$ | $0.316 - 0.327$ |
| **$\gamma$** | Susceptibility $\chi \propto \|t\|^{-\gamma}$ | $1$ | $7/4 = 1.75$ | $1.2405 \pm 0.0015$ | $1.23 - 1.25$ |
| **$\delta$** | Critical isotherm $M \propto h^{1/\delta}$ | $3$ | $15$ | $4.82(4)$ | $4.6 - 4.9$ |
| **$\nu$** | Correlation length $\xi \propto \|t\|^{-\nu}$ | $1/2 = 0.5$ | $1$ | $0.630(2)$ | $0.625 \pm 0.010$ |
| **$\eta$** | Critical correlation $G(r) \propto r^{-(d-2+\eta)}$ | $0$ | $1/4 = 0.25$ | $0.032 \pm 0.003$ | $0.016 - 0.06$ |

---

### Homogeneous Functions and Static Scaling Identities
A function $f(r)$ is **homogeneous** of degree $p$ if rescaling its argument by $\lambda$ satisfies $f(\lambda r) = \lambda^p f(r)$. More generally, the **static scaling hypothesis** asserts that the singular part of the free energy density $f_s(t, h)$ is a **generalized homogeneous function**:
$$\lambda f_s(t, h) = f_s\!\left(\lambda^{a_t} t, \, \lambda^{a_h} h\right)$$
Because all thermodynamic observables ($M = -\partial f_s / \partial h$, $\chi = -\partial^2 f_s / \partial h^2$, $C_V \propto -\partial^2 f_s / \partial t^2$) derive from just **two** independent scaling exponents ($a_t$ and $a_h$, or equivalently $\nu$ and $\eta$), the six critical exponents $\alpha, \beta, \gamma, \delta, \nu, \eta$ are linked by **four exact scaling laws**:

1. **Rushbrooke's Identity:**
 $$\alpha + 2\beta + \gamma = 2$$
2. **Widom's Identity:**
 $$\gamma = \beta(\delta - 1) \quad\Longleftrightarrow\quad \delta = 1 + \frac{\gamma}{\beta}$$
3. **Fisher's Identity:**
 $$\gamma = \nu(2 - \eta) \quad\Longleftrightarrow\quad \frac{\gamma}{\nu} = 2 - \eta$$
4. **Josephson's Hyperscaling Identity** (involves spatial dimension $d$; holds for $d \le d_u = 4$):
 $$d\nu = 2 - \alpha$$

---

### Universality: Why Microscopic Details Vanish at $T_c$
Because $\xi \to \infty$ at $T_c$, cooperative fluctuations average over millions of lattice constants and **forget microscopic details** (such as lattice geometry—square vs. Triangular, or simple cubic vs. BCC—and bond strength $J$).
- **Universal quantities** (depend *only* on spatial dimensionality $d$, order-parameter symmetry/dimensionality $n$, and interaction range): **critical exponents ($\alpha, \beta, \gamma, \delta, \nu, \eta$)** and **dimensionless scaling functions** (including the critical Binder cumulant $U_4^*$).
- **Non-universal quantities** (depend on lattice coordination number $z_{\text{coord}}$, microscopic coupling $J$, and next-nearest-neighbor interactions): **$T_c$, $P_c$, $\rho_c$**, and dimensionful critical amplitudes!
- Thus, a **3D uniaxial magnet ($n=1$)**, a **3D classical liquid–vapor critical point** (order parameter $\rho_L - \rho_G$, scalar $n=1$), and a **3D binary alloy unmixing transition** all belong to the **3D Ising ($d=3, n=1$) universality class** and share the exact same exponents ($\beta \approx 0.325, \gamma \approx 1.24, \nu \approx 0.63$)!


In [13]:
ws.render_section('q23_09')

**Q9 · easy** · 💡

Which two fundamental properties of a physical system with short-range interactions determine its **static universality class** (and therefore its exact critical exponents $\alpha, \beta, \gamma, \delta, \nu, \eta$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall what remains after coarse-graining over diverging correlation lengths $\xi \to \infty$.

</div>
</details>

- [ ] **(A)** The lattice coordination number $z_{\text{coord}}$ and the magnitude of the exchange coupling $J$.
- [ ] **(B)** The spatial dimensionality $d$ and the symmetry (number of components $n$) of the order parameter.
- [ ] **(C)** The critical temperature $T_c$ and the critical pressure $P_c$.
- [ ] **(D)** The particle mass $m$ and the choice of Kadanoff block-spin size $b$.

In [14]:
ws.render_section('q23_10')

**Q10 · medium** · 💡

Which of the following quantities are **universal** (identical for a 3D simple-cubic Ising model and a 3D body-centered-cubic Ising model, as well as a 3D classical liquid–gas transition)? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Identify which properties depend strictly on long-wavelength fixed-point scale invariance versus those that reflect microscopic non-universal details like coupling constants or coordination geometries.

</div>
</details>

- [ ] **(A)** The correlation-length critical exponent $\nu$ and order-parameter exponent $\beta$.
- [ ] **(B)** The dimensionless critical coupling $K_c = J / (k_B T_c)$ and critical temperature $T_c$.
- [ ] **(C)** The susceptibility exponent $\gamma$ and anomalous dimension $\eta$.
- [ ] **(D)** Dimensionless finite-size scaling functions and the critical Binder cumulant ratio $U_4^*$ (for matching boundary conditions).

In [15]:
ws.render_section('q23_11')

**Q11 · easy** · 💡

For the 2D Ising universality class, the exact order-parameter exponent is $\beta = 1/8 = 0.125$ and the susceptibility exponent is $\gamma = 7/4 = 1.75$. Using **Widom's scaling hypothesis**, calculate the exact critical isotherm exponent $\delta$ governing the field dependence of the magnetization ($M \propto h^{1/\delta}$ at $T = T_c$). *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the critical isotherm exponent to the order parameter and magnetic susceptibility exponents via Widom scaling.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q23_12')

**Q12 · medium** · 💡

For a 3D Ising system ($d = 3$), high-precision Monte Carlo simulations give $\beta = 0.325$ and $\gamma = 1.240$. Using **Rushbrooke's scaling relation**, calculate the specific-heat critical exponent $\alpha$. *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the thermodynamic scaling inequality relating the specific heat, spontaneous magnetization, and susceptibility exponents at equality.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q23_13')

**Q13 · medium** · 💡

Using **Josephson's hyperscaling relation** in $d = 3$ spatial dimensions with specific-heat exponent $\alpha = 0.110$, compute the correlation-length critical exponent $\nu$. *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Invoke the hyperscaling relation connecting spatial dimensionality, the divergence of the correlation length, and the singularity in the singular free energy density.

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q23_14')

**Q14 · medium** · 💡

Using **Fisher's scaling relation**, determine the exact anomalous dimension $\eta$ of the 2D Ising model given correlation-length exponent $\nu = 1$ and susceptibility exponent $\gamma = 7/4 = 1.75$. *(Enter as a decimal.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the spatial decay of the two-point critical correlation function to the divergence of the static susceptibility.

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('q23_15')

**Q15 · hard** · 💡

In Mean-Field (Landau) theory, the critical exponents are $\alpha = 0, \beta = 1/2, \gamma = 1, \delta = 3, \nu = 1/2, \eta = 0$ regardless of dimension. At what **upper critical spatial dimension** $d_u$ does Josephson hyperscaling become consistent with these Mean-Field values? *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the spatial dimension at which the singular free energy density volume scaling matches the mean-field specific-heat behavior.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q23_16')

**Q16 · medium** · 💡

For the generalized homogeneous Gibbs free energy $f(\lambda^{a_t} t, \lambda^{a_h} h) = \lambda f(t, h)$, differentiating with respect to $h$ gives the magnetization scaling relation $\lambda^{a_h} M(\lambda^{a_t} t, \lambda^{a_h} h) = \lambda M(t, h)$. Setting $h = 0$ and choosing $\lambda = (-t)^{-1/a_t}$ for $t < 0$, how is the order-parameter critical exponent $\beta$ expressed in terms of $a_t$ and $a_h$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the scaling factor $\lambda$ is chosen to eliminate the reduced temperature dependence inside the homogeneous scaling function, and read off the resulting power of $(-t)$.

</div>
</details>

- [ ] **(A)** $\beta = \frac{1 - a_h}{a_t}$
- [ ] **(B)** $\beta = \frac{a_t}{a_h}$
- [ ] **(C)** $\beta = a_t + a_h$
- [ ] **(D)** $\beta = \frac{2a_h - 1}{a_t}$

In [21]:
ws.render_section('sec_kadanoff_wilson_rg')

### Kadanoff's Block-Spin Coarse-Graining (1966)
To understand how macroscopic scale invariance emerges at $T_c$, **Leo Kadanoff** proposed grouping the lattice into blocks of linear size $b$ (for example, $b \times b = 2 \times 2$ squares containing $b^d = 4$ spins in 2D) and replacing each block $I$ by a single **block spin (superspin)** $S_I' = \pm 1$ via a majority rule (flipping a fair coin to break ties when $\sum_{i \in I} s_i = 0$):
$$S_I' = \text{sign}\!\left(\sum_{i \in I} s_i\right)$$
Next, we rescale all spatial lengths by the blocking factor $b$ so that the new lattice spacing is restored to $a' = a$:
$$L' = \frac{L}{b}, \qquad \xi' = \frac{\xi}{b}$$

---

### Wilson's Renormalization Group (RG) Flow in Coupling Space (1971)
Integrating out the short-wavelength intra-block degrees of freedom maps the original dimensionless Hamiltonian $H = \sum_\alpha K_\alpha S_\alpha$ (where $K_1 = J_{\text{nn}} / (k_B T)$ is the nearest-neighbor coupling, $K_2 = J_{\text{nnn}} / (k_B T)$ is the next-nearest-neighbor coupling, etc.) into a renormalized Hamiltonian $H' = R(H)$ with renormalized couplings:
$$\mathbf{K}' = R(\mathbf{K}) \quad\text{or in a 1-parameter projection:}\quad K' = R(K)$$

Since every RG step divides the correlation length by $b > 1$ ($\xi(K') = \xi(K) / b$), repeated coarse-graining $K^{(n+1)} = R(K^{(n)})$ creates a **Renormalization Group flow**:
1. **High-Temperature Disordered Sink ($K = 0 \iff T = \infty$):**
 If we start at $T > T_c$ ($K_0 < K^*$), $\xi_0$ is finite. Each blocking step shrinks $\xi^{(n)} = \xi_0 / b^n \to 0$, driving $K^{(n)} \to 0$ (where spins look completely random). Thus **$K = 0$ is a trivial stable fixed point**.
2. **Low-Temperature Ordered Sink ($K = \infty \iff T = 0$):**
 If we start at $T < T_c$ ($K_0 > K^*$), domains are ordered and majority rule locks block spins into alignment, driving $K^{(n)} \to \infty$. Thus **$K = \infty$ is a trivial stable fixed point**.
3. **Non-Trivial Unstable Critical Fixed Point ($K^* = R(K^*) \iff T = T_c$):**
 A fixed point $K^* = R(K^*)$ requires $\xi(K^*) = \xi(K^*) / b$, which has only two solutions: $\xi = 0$ (trivial sinks) or **$\xi = \infty$ (the critical point $T = T_c$)**! At $K^*$, configurations look statistically identical before and after blocking (**scale invariance**). Because any tiny perturbation $\delta K = K_0 - K^* \neq 0$ flows away toward $0$ or $\infty$, **$K^*$ is an unstable fixed point** along the thermal direction.

---

### Extracting the Critical Exponent $\nu$ from the Linearized RG Map
Near the critical fixed point $K^*$, we linearize the recursion relation $K' = R(K)$ by a Taylor expansion:
$$K' - K^* \approx \left.\frac{dR}{dK}\right|_{K^*} (K - K^*) \equiv \lambda_T (K - K^*) = b^{y_T} (K - K^*)$$
where $\lambda_T = \left.\frac{dR}{dK}\right|_{K^*} > 1$ is the thermal RG eigenvalue and $y_T = \frac{\ln \lambda_T}{\ln b} > 0$ is the **thermal scaling exponent**.
Because the correlation length transforms as $\xi(K) = b\,\xi(K')$ while diverging as $\xi(K) \propto |K - K^*|^{-\nu}$, substituting $K' - K^* = b^{y_T}(K - K^*)$ yields:
$$|K - K^*|^{-\nu} = b \left|b^{y_T}(K - K^*)\right|^{-\nu} = b^{1 - y_T \nu} |K - K^*|^{-\nu} \implies 1 - y_T \nu = 0$$
Therefore, the correlation-length critical exponent $\nu$ is determined directly by the derivative of the RG transformation at the fixed point:
$$\nu = \frac{1}{y_T} = \frac{\ln b}{\ln\!\left(\left.\frac{dR}{dK}\right|_{K^*}\right)}$$

In the full multi-coupling space $(K_1, K_2, K_3, \dots)$ (Figure 9.3, of the reference notes):
- **Relevant eigendirections ($y_i > 0$, $\lambda_i = b^{y_i} > 1$):** grow under RG (temperature $t$ with $y_T = 1/\nu > 0$ and magnetic field $h$ with $y_h = \beta\delta/\nu > 0$); must be tuned to zero to hit criticality.
- **Irrelevant eigendirections ($y_i < 0$, $\lambda_i = b^{y_i} < 1$):** decay to zero ($b^{-|y_i| n} \to 0$) under repeated blocking! Different microscopic Hamiltonians on the **critical surface** flow into the **same fixed point $\mathbf{K}^*$**, providing the mathematical proof of **Universality**!


In [22]:
ws.render_section('concept_kadanoff_rg_flow')

### Interactive Explorer: Kadanoff Block-Spin RG Flow & Cobweb Iteration

Adjust the initial dimensionless coupling $K_0 = J/(k_B T)$, the number of RG blocking steps $n$, and the block size $b$ to trace the cobweb trajectory $K^{(n+1)} = R(K^{(n)})$ relative to the unstable critical fixed point $K^* \approx 0.441$.

**Key Takeaway:** Starting anywhere below $K^*$ ($T > T_c$) staircases down to the high-temperature sink $K=0$; starting above $K^*$ ($T < T_c$) staircases up to the $T=0$ sink $K=\infty$. The slope $\lambda_T = dR/dK|_{K^*} = b^{1/\nu}$ at the unstable fixed point $K^*$ controls how rapidly trajectories diverge from criticality.

In [23]:
ws.render_section('q23_17')

**Q17 · easy** · 💡

When a $2\times 2$ Kadanoff block-spin transformation ($b = 2$) is applied once to a configuration with correlation length $\xi$ (measured in units of the lattice spacing), what is the renormalized correlation length $\xi'$ of the blocked lattice (measured in units of the new block-lattice spacing)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall how lengths are rescaled when $b\times b$ sites are grouped into 1 supersite.

</div>
</details>

- [ ] **(A)** $\xi' = 2\xi$
- [ ] **(B)** $\xi' = \xi / 2$
- [ ] **(C)** $\xi' = \xi^2$
- [ ] **(D)** $\xi' = \xi - 2$

In [24]:
ws.render_section('q23_18')

**Q18 · easy** · 💡

Why must any Renormalization Group fixed point $K^* = R(K^*)$ have a dimensionless correlation length equal to either $\xi = 0$ or $\xi = \infty$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how length scales transform under spatial coarse-graining when evaluated at an invariant fixed point of the transformation.

</div>
</details>

- [ ] **(A)** Because at $K' = K^*$, the relation $\xi(K^*) = \xi(K^*) / b$ for $b > 1$ is only satisfied if $\xi(K^*) = 0$ (trivial fixed points at $T=0, \infty$) or $\xi(K^*) = \infty$ (critical fixed point at $T = T_c$).
- [ ] **(B)** Because the partition function $Z$ vanishes at every fixed point.
- [ ] **(C)** Because Kadanoff blocking deletes all nearest-neighbor interactions when $\xi$ is finite.
- [ ] **(D)** Because the Ising model only exists at $T = 0$ and $T = \infty$.

In [25]:
ws.render_section('q23_19')

**Q19 · medium** · 💡

Suppose we initialize a 2D Ising lattice at a temperature slightly *above* the critical temperature, $T_0 = 1.05\\,T_c$ (so $K_0 < K^*$), and repeatedly apply Kadanoff $2\times 2$ block-spin transformations ($n = 1, 2, 3, \dots$). How do the effective coupling $K^{(n)}$ and visual spin snapshots evolve?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Trace the direction of renormalization flow for initial couplings weaker than the critical fixed-point coupling and observe how effective correlations evolve across successive blockings.

</div>
</details>

- [ ] **(A)** $K^{(n)}$ flows toward $0$ ($T_{\text{eff}} \to \infty$), and the spin snapshots look progressively more random with smaller correlated patches ($\xi^{(n)} = \xi_0 / 2^n \to 0$).
- [ ] **(B)** $K^{(n)}$ flows toward $\infty$ ($T_{\text{eff}} \to 0$), and the spin snapshots lock into a uniform all-up or all-down ferromagnetic state.
- [ ] **(C)** $K^{(n)}$ converges toward the critical fixed point $K^*$, and the snapshots develop fractal critical domains.
- [ ] **(D)** $K^{(n)}$ oscillates periodically between $0$ and $\infty$.

In [26]:
ws.render_section('q23_20')

**Q20 · medium** · 💡

An approximate $b = 2$ Kadanoff block-spin RG transformation for the 2D Ising model yields a critical fixed point $K^* = R(K^*)$ where the linearized thermal eigenvalue of the recursion map is $\lambda_T = \left.\frac{dR}{dK}\right|_{K^*} = 1.95$. Calculate the resulting RG estimate for the **correlation-length critical exponent** $\nu$. *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Relate the thermal eigenvalue of the linearized RG flow to the spatial rescaling factor and the correlation length exponent.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q23_21')

**Q21 · medium** · 💡

Conversely, if an exact Renormalization Group transformation with spatial rescaling factor $b = 3$ is applied to the 2D Ising model (where the exact correlation-length exponent is $\nu = 1.0$), what must the thermal eigenvalue $\lambda_T$ equal at the critical fixed point? *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Invert the connection between the linearized RG eigenvalue, the spatial scaling factor, and the critical exponent $\nu$.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q23_22')

**Q22 · medium** · 💡

When a $2\times 2$ block-spin transformation is applied to a 2D Ising Hamiltonian containing *only* nearest-neighbor interactions ($K_1 S_1$), what happens to the structure of the renormalized Hamiltonian $H_{\text{new}} = R(H)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider whether neighboring blocks interact only through a single nearest-neighbor coupling or also generate diagonal and multi-spin couplings.

</div>
</details>

- [ ] **(A)** $H_{\text{new}}$ remains a pure nearest-neighbor Ising model with the exact same coupling $K_1' = K_1$ at all temperatures.
- [ ] **(B)** Coarse-graining generates longer-ranged and multi-spin effective couplings ($H_{\text{new}} = \sum_\alpha K_\alpha S_\alpha$, including next-nearest-neighbor $K_2$, 4-spin plaquette terms, etc.), so the RG flows in a multi-dimensional coupling space.
- [ ] **(C)** All spin interactions vanish after a single blocking step.
- [ ] **(D)** The discrete Ising spins $S_I' = \pm 1$ are converted into continuous $O(3)$ Heisenberg vectors.

In [29]:
ws.render_section('q23_23')

**Q23 · hard** · 💡

In the two-coupling RG flow diagram $(K_1, K_2)$ for the 2D Ising model with nearest-neighbor ($K_1$) and next-nearest-neighbor ($K_2$) interactions, there is a curve (the **critical surface**) passing through the Critical Fixed Point (FP) that separates trajectories flowing to the High-Temperature FP $(0,0)$ from those flowing to the Low-Temperature FP $(\infty, \infty)$. How does this flow diagram explain **Universality**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Follow the arrows along the critical surface into the Critical FP vs. The arrows leaving the Critical FP perpendicular to the critical surface.

</div>
</details>

- [ ] **(A)** Along the critical surface, trajectories from models with different microscopic ratios $K_2 / K_1$ all flow *into* the same Critical FP (irrelevant eigenvalue $\lambda_2 < 1$), so they share the exact same linearized unstable eigenvalue $\lambda_T > 1$ and critical exponents!
- [ ] **(B)** The critical surface shows that $T_c$ is identical regardless of whether $K_2$ is zero or nonzero.
- [ ] **(C)** Every point on the $(K_1, K_2)$ plane is a fixed point with $\xi = \infty$.
- [ ] **(D)** Trajectories on the critical surface flow directly to $(K_1, K_2) = (0, 0)$.

In [30]:
ws.render_section('q23_24')

**Q24 · hard** · 💡

In **Monte Carlo Renormalization Group (MCRG)**, how do we determine $T_c$ directly from Monte Carlo simulations without knowing the analytical recursion equation $R(K)$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At a fixed point $H^* = R(H^*)$, compare correlation functions measured after $n$ blockings vs. $n+1$ blockings on matching renormalized lattice sizes.

</div>
</details>

- [ ] **(A)** Find the temperature $T$ at which correlation functions of configurations blocked $n$ times and $n+1$ times (at matching renormalized lattice size) are identical.
- [ ] **(B)** Find the temperature $T$ where the single-spin Metropolis acceptance ratio equals $100\\%$.
- [ ] **(C)** Find the temperature $T$ where the internal energy $U(T)$ equals zero.
- [ ] **(D)** Run at $T = 0$ and count how many blocking steps are needed to reach $L = 1$.

In [31]:
ws.render_section('sec_finite_size_scaling')

### Why Finite Systems Cannot Have True Phase Transitions
In any finite simulation box of volume $V = L^d$ with $N = L^d$ spins, the canonical partition function
$$Z_L(T, h) = \sum_{\{s_i = \pm 1\}} \exp\!\left(-\frac{H(\{s_i\})}{k_B T}\right)$$
is a **finite sum of $2^N$ smooth, strictly positive analytic exponentials**. A finite sum of analytic functions is analytic for all $T > 0$! Therefore:
- No thermodynamic derivative ($C_V(T, L)$ or $\chi(T, L)$) can truly diverge to $\infty$ in a finite box.
- Instead, when the bulk correlation length $\xi_{\infty}(t) \propto |t|^{-\nu}$ grows comparable to the box size $L$ ($\xi_{\infty} \sim L$, i.e. $|t| \sim L^{-1/\nu}$), singularities are **rounded into smooth peaks** and the apparent pseudocritical peak temperature $T_c(L)$ is **shifted** relative to the bulk $T_c(\infty)$.

---

### Deriving the Finite-Size Scaling (FSS) Ansatz
Rather than treating finite-size rounding as a nuisance, **Finite-Size Scaling (FSS)** *exploits* the dependence on $L$ to extract bulk critical exponents!
Near $T_c$, a single length scale $\xi \propto |t|^{-\nu}$ governs correlations, so the dimensionless ratio controlling finite-size effects is:
$$\frac{L}{\xi(t)} \propto L |t|^{\nu} = \left(|t|\,L^{1/\nu}\right)^{\nu}$$
Replacing $|t| \sim L^{-1/\nu}$ in the bulk power laws at $T \approx T_c$ gives the peak scalings with box size $L$:
- **Magnetization at $T_c$:**
 $$M(t=0, L) \propto \left.\left(|t|^\beta\right)\right|_{|t| \sim L^{-1/\nu}} = L^{-\beta/\nu}$$
- **Susceptibility Peak Height:**
 $$\chi_{\max}(L) \propto \left.\left(|t|^{-\gamma}\right)\right|_{|t| \sim L^{-1/\nu}} = L^{\gamma/\nu}$$
- **Specific Heat Peak Height:**
 $$C_{\max}(L) \propto \left.\left(|t|^{-\alpha}\right)\right|_{|t| \sim L^{-1/\nu}} = L^{\alpha/\nu} \quad (\text{or } \propto \ln L \text{ when } \alpha = 0 \text{ in 2D Ising})$$
- **Pseudocritical Peak Shift $T_c(L)$:**
 The susceptibility peak occurs at a fixed value $x_{\text{peak}}$ of the scaled variable $x = t L^{1/\nu}$, so:
 $$T_c(L) - T_c(\infty) \propto L^{-1/\nu}$$

More generally, writing the free energy as $F_N = L^y f\!\left(t L^{1/\nu}, \, h L^{\beta\delta/\nu}\right)$ and differentiating yields the **full Finite-Size Scaling equations** (at $h = 0$):
$$M(t, L) = L^{-\beta/\nu}\,\tilde{m}\!\left(t L^{1/\nu}\right)$$
$$\chi(t, L) = L^{\gamma/\nu}\,\tilde{\chi}\!\left(t L^{1/\nu}\right)$$
$$\xi(t, L) = L\,\tilde{\xi}\!\left(t L^{1/\nu}\right)$$

---

### Universal Data Collapse and Dimensionless Crossing Ratios
The FSS equations predict two powerful graphical diagnostics:
1. **Correlation Length Ratio Crossing ($\xi(T, L) / L$ or $L / \xi(T, L)$):**
 Because $\xi(t, L) / L = \tilde{\xi}(t L^{1/\nu})$ has **zero prefactor power of $L$**, at $t = 0$ ($T = T_c$) we have $\xi(T_c, L)/L = \tilde{\xi}(0) = \text{const}$ independent of $L$. Thus curves of $\xi(T, L)/L$ (or $L/\xi(T, L)$) plotted vs. $T$ for different box sizes $L_1 < L_2 < L_3$ **all intersect at $T = T_c$**!
2. **Universal Data Collapse:**
 If we plot the rescaled magnetization **$M(t, L)\,L^{\beta/\nu}$** (or rescaled susceptibility **$\chi(t, L)\,L^{-\gamma/\nu}$**) on the vertical axis against the rescaled temperature **$x = t\,L^{1/\nu}$** on the horizontal axis, data from **all system sizes $L$ collapse onto a single universal curve** $\tilde{m}(x)$ (or two branches $\tilde{m}_\pm(|x|)$ for $T < T_c$ and $T > T_c$ on a log-log plot)!


In [32]:
ws.render_section('concept_finite_size_scaling_collapse')

### Interactive Explorer: Finite-Size Scaling & Universal Data Collapse

Switch between the unscaled finite-size susceptibility curves $\chi(T, L)$ for $L \in \\{8, 16, 32, 64\\}$ and the rescaled FSS collapse plot $\chi L^{-\gamma/\nu}$ vs. $t L^{1/\nu}$. Tune the trial exponents $\nu_{\text{trial}}$ and $\gamma_{\text{trial}}$ to achieve a single master curve!

**Key Takeaway:** In the raw view, doubling $L$ in the 2D Ising model increases the peak height $\chi_{\max}(L) \propto L^{\gamma/\nu} = L^{1.75}$ by $2^{1.75} \approx 3.36\times$ and narrows the critical region by $2^{-1/\nu} = 0.5\times$. In the rescaled view, the curves collapse onto a single universal scaling function $\tilde{\chi}(x)$ only when $\nu = 1.00$ and $\gamma = 1.75$.

In [33]:
ws.render_section('q23_25')

**Q25 · easy** · 💡

Why can a canonical partition function $Z_L(T) = \sum_{\\{s_i\\}} e^{-H/k_B T}$ on a **finite** lattice of $N = L^d$ spins never exhibit a true mathematical singularity (infinite derivative) in $\chi(T, L)$ or $C_V(T, L)$ at any nonzero temperature $T > 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how many terms appear in the partition sum $Z_L(T)$ when $N < \infty$.

</div>
</details>

- [ ] **(A)** Because $Z_L(T)$ is a finite sum of $2^N$ smooth, strictly positive analytic exponential functions, so its logarithm and all finite-order temperature derivatives are smooth analytic functions of $T$.
- [ ] **(B)** Because periodic boundary conditions set the net magnetization identically to zero for every microstate.
- [ ] **(C)** Because the Metropolis algorithm violates detailed balance in finite boxes.
- [ ] **(D)** Because the correlation length $\xi$ is infinite at all temperatures in a finite box.

In [34]:
ws.render_section('q23_26')

**Q26 · medium** · 💡

In a finite-size scaling study of the 2D Ising model ($\nu = 1$, $\gamma = 7/4 = 1.75$), how many times larger is the maximum magnetic susceptibility peak $\chi_{\max}(L=64)$ in a $64 \times 64$ lattice compared to $\chi_{\max}(L=16)$ in a $16 \times 16$ lattice? *(Round to 2 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the finite-size scaling relation for the peak susceptibility as a function of box size and critical exponents.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q23_27')

**Q27 · medium** · 💡

At the critical point $T = T_c$ of the 2D Ising model ($\beta = 1/8 = 0.125$, $\nu = 1$), how does the average absolute magnetization per spin scale with box size? If $\langle |M| \rangle_{T_c} = 0.600$ at $L = 10$, what is the predicted value of $\langle |M| \rangle_{T_c}$ at $L = 160$? *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the finite-size scaling law for the order parameter right at the critical temperature where the correlation length is bounded by the system dimension.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('q23_28')

**Q28 · hard** · 💡

In a finite-size scaling experiment on a 3D Ising magnet, the susceptibility peak occurs at $T_c(L=10) = 4.5515$ for $L = 10$ and at $T_c(L=20) = 4.5248$ for $L = 20$, while the true bulk critical temperature is $T_c(\infty) = 4.5115$. Based on the finite-size shift scaling of pseudocritical temperatures, estimate the correlation-length exponent $\nu$ from these two box sizes. *(Round to 2 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Form the ratio of the pseudocritical temperature displacements relative to the bulk critical value for both lattice dimensions and solve for the scaling exponent.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('q23_29')

**Q29 · medium** · 💡

In finite-size scaling analyses of the 2D Ising model, magnetization data across multiple lattice sizes $L$ are plotted on a log-log graph of $M L^{\beta/\nu}$ vs. $x = L^{1/\nu} |t|$. Why does the collapsed data split into **two universal branches** with asymptotic log-log slopes $+1/8$ (for $T < T_c$) and $-7/8$ (for $T > T_c$) at large $x \gg 1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the physical requirement that the magnetization in a finite box must smoothly match the size-independent thermodynamic limit when the box dimension far exceeds the correlation length.

</div>
</details>

- [ ] **(A)** Because for $T < T_c$ and $x \gg 1$, $\tilde{m}_-(x) \propto x^{\beta} = x^{1/8}$ cancels the $L^{\beta/\nu}$ factor to recover the $L$-independent bulk magnetization $M \propto (-t)^{1/8}$; whereas for $T > T_c$, $\langle |M| \rangle \propto L^{-1} |t|^{-\gamma/2} = L^{-1/8} x^{-7/8}$.
- [ ] **(B)** Because the critical exponent $\beta$ is $+1/8$ below $T_c$ and $-7/8$ above $T_c$ in an infinite system.
- [ ] **(C)** Because the 2D Ising model has two distinct critical temperatures.
- [ ] **(D)** Because periodic boundary conditions break translational invariance above $T_c$.

In [38]:
ws.render_section('q23_30')

**Q30 · medium** · 💡

Consider the ratio $L / \xi(K, L)$ plotted against coupling $K = J/(k_B T)$ for three system sizes $L_1 < L_2 < L_3$. As $L$ increases from $L_1$ to $L_3$, how does $L / \xi(K, L)$ behave for $K < K^*$ ($T > T_c$, disordered) versus $K > K^*$ ($T < T_c$, ordered), and what happens at $K = K^*$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how the ratio of system size to correlation length behaves when the correlation length saturates at a finite value versus when it is constrained by the system boundaries at criticality.

</div>
</details>

- [ ] **(A)** For $K < K^*$ ($T > T_c$), $L/\xi$ increases with $L$; at $K = K^*$ ($T = T_c$), all three curves cross at a single universal point because $\xi(K^*, L) \propto L$; and for $K > K^*$, the curves fan out in the opposite order.
- [ ] **(B)** All three curves are parallel horizontal lines that never intersect.
- [ ] **(C)** The curves intersect at $K = 0$ ($T = \infty$) rather than $K^*$.
- [ ] **(D)** For $K < K^*$, $L/\xi$ decreases to zero as $L \to \infty$.

In [39]:
ws.render_section('q23_31')

**Q31 · medium** · 💡

In the 2D Ising model, the specific heat exponent is $\alpha = 0$, corresponding to a logarithmic divergence $C_V(t) \propto -\ln|t|$ in the thermodynamic limit as $t \to 0$. According to finite-size scaling, how does the specific heat peak height $C_{\max}(L)$ scale with linear box size $L$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Replace the bulk thermal distance to criticality with the effective cutoff imposed by the finite box size.

</div>
</details>

- [ ] **(A)** $C_{\max}(L) \propto \ln L$
- [ ] **(B)** $C_{\max}(L) \propto L^2$
- [ ] **(C)** $C_{\max}(L) \propto L^{-1}$
- [ ] **(D)** $C_{\max}(L)$ decreases exponentially with $L$.

In [40]:
ws.render_section('sec_binder_cumulant')

### Evolution of the Order-Parameter Distribution $P_L(M)$ Across $T_c$
Tracking the full probability distribution $P_L(M)$ of the magnetization per spin $M = \frac{1}{N}\sum_i s_i$ in a finite box of size $L$ reveals how symmetry breaking occurs:**
 The lattice of volume $L^d$ decomposes into roughly $(L / \xi)^d \gg 1$ statistically independent blocks of volume $\xi^d$. By the **Central Limit Theorem (CLT)**, their average $M$ obeys a **zero-mean Gaussian distribution** of variance $\sigma_M^2 = \langle M^2 \rangle = k_B T \chi / L^d \propto L^{-d}$:
 $$P_L(M) \simeq \frac{1}{\sqrt{2\pi\sigma_M^2}} \exp\!\left(-\frac{M^2}{2\sigma_M^2}\right) \qquad (T > T_c)$$
2. **Critical Point ($T = T_c$, where $\xi \sim L$):**
 Because $\xi$ spans the entire simulation box, the Central Limit Theorem breaks down! $P_L(M)$ becomes a **broad, non-Gaussian double-peaked/flat-topped universal scaling distribution** whose width scales as $L^{-\beta/\nu}$:
 $$P_L(M) = L^{\beta/\nu}\,\tilde{P}\!\left(M\,L^{\beta/\nu}\right) \qquad (T = T_c)$$
3. **Low-Temperature Ferromagnetic Phase ($T < T_c$, where $L \gg \xi$):**
 Spontaneous symmetry breaking splits $P_L(M)$ into **two sharp symmetric peaks** centered at $\pm M_0(T)$ (since $h = 0$ preserves $M \leftrightarrow -M$ symmetry in a finite ergodic run):
 $$P_L(M) \simeq \frac{1}{2}\left[\delta(M - M_0) + \delta(M + M_0)\right] \qquad (T < T_c, \; L \to \infty)$$

---

### Binder's 4th-Order Cumulant $U_4(T, L)$
To distinguish a Gaussian distribution from a bimodal distribution *and* cancel out unknown critical exponents, **Kurt Binder (1981)** introduced the reduced fourth-order cumulant (related to statistical kurtosis, of the reference notes):
$$U_4(T, L) \equiv 1 - \frac{\langle M^4 \rangle}{3\,\langle M^2 \rangle^2}$$

Let us evaluate $U_4(T, L)$ in all three regimes as $L \to \infty$:
1. **Disordered Phase ($T > T_c$): $U_4 \to 0$**
 For any zero-mean Gaussian distribution $P(M) \propto e^{-M^2/(2\sigma^2)}$, **Wick's theorem** (Gaussian moment identity) states that the fourth moment is exactly three times the square of the second moment:
 $$\langle M^4 \rangle = 3\sigma^4 = 3\,\langle M^2 \rangle^2 \implies U_4(T > T_c, L \to \infty) = 1 - \frac{3\langle M^2 \rangle^2}{3\langle M^2 \rangle^2} = 0$$
2. **Ordered Phase ($T < T_c$): $U_4 \to 2/3$**
 For two narrow symmetric delta peaks at $M = \pm M_0$, both even moments are fixed by $M_0$:
 $$\langle M^2 \rangle \to M_0^2, \qquad \langle M^4 \rangle \to M_0^4 = \langle M^2 \rangle^2 \implies U_4(T < T_c, L \to \infty) = 1 - \frac{M_0^4}{3(M_0^2)^2} = 1 - \frac{1}{3} = \frac{2}{3} \approx 0.6667$$
3. **Critical Point ($T = T_c$): Scale-Invariant Universal Crossing!**
 By Finite-Size Scaling, the $k$-th moment of the magnetization scales as $\langle M^k \rangle = L^{-k\beta/\nu}\,\tilde{m}_k(t L^{1/\nu})$. Therefore, in the ratio $\frac{\langle M^4 \rangle}{\langle M^2 \rangle^2}$, the prefactor powers of $L$ **cancel out identically**:
 $$\frac{\langle M^4 \rangle}{\langle M^2 \rangle^2} = \frac{L^{-4\beta/\nu}\,\tilde{m}_4(t L^{1/\nu})}{\left[L^{-2\beta/\nu}\,\tilde{m}_2(t L^{1/\nu})\right]^2} = \frac{\tilde{m}_4(t L^{1/\nu})}{\left[\tilde{m}_2(t L^{1/\nu})\right]^2} \implies U_4(t, L) = \tilde{U}_4\!\left(t L^{1/\nu}\right)$$
 At $t = 0$ ($T = T_c$), $U_4(T_c, L) = \tilde{U}_4(0) \equiv U_4^*$ is **independent of box size $L$**!
 - Consequently, plotting $U_4(T, L)$ vs. $T$ for several lattice sizes ($L = 10, 20, 40, \dots$) produces a family of curves that **cross at a single point $T = T_c$**!
 - Crucially, this locates $T_c$ with high precision **without needing any prior value for $\beta$ or $\nu$**!
 - Moreover, the slope of $U_4(T, L)$ with respect to $T$ at the crossing point scales as:
 $$\left.\frac{\partial U_4}{\partial T}\right|_{T_c} \propto L^{1/\nu}$$
 providing a direct way to extract $\nu$ from the ratio of slopes of different $L$ curves at $T_c$!


In [41]:
ws.render_section('concept_binder_cumulant_crossing')

### Interactive Explorer: Binder's 4th-Order Cumulant Crossing at T_c

Slide the probe temperature $T$ across the critical region of the 2D Ising model ($T_c \approx 2.269\\,J/k_B$) to observe how $U_4(T, L)$ approaches $2/3$ below $T_c$, approaches $0$ above $T_c$, and intersects at a single size-independent value $U_4^* \approx 0.611$ right at $T_c$.

**Key Takeaway:** Because the $L^{-4\beta/\nu}$ prefactors in $\langle M^4 \rangle$ and $\langle M^2 \rangle^2$ cancel identically, $U_4(T, L) = \tilde{U}_4(t L^{1/\nu})$ has zero prefactor exponent. All box sizes cross at $t = 0$ ($T = T_c$) without requiring any trial values of $\beta$ or $\nu$!

In [42]:
ws.render_section('q23_32')

**Q32 · easy** · 💡

In the high-temperature disordered phase ($T > T_c$) as $L \to \infty$, the Central Limit Theorem ensures that the magnetization distribution $P_L(M)$ approaches a zero-mean Gaussian over independent spatial correlation volumes. What is the asymptotic value of **Binder's fourth-order cumulant** $U_4 = 1 - \frac{\langle M^4 \rangle}{3\langle M^2 \rangle^2}$ in this disordered thermodynamic limit? *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the relationship between the second and fourth moments for any Gaussian distributed random variable with zero mean.

</div>
</details>

_Numeric answer required._

In [43]:
ws.render_section('q23_33')

**Q33 · easy** · 💡

In the low-temperature ferromagnetic phase ($T < T_c, h = 0$) as $L \to \infty$, the magnetization distribution $P_L(M)$ sharpens into two symmetric delta-like peaks at $\pm M_{\text{spont}}$. What is the asymptotic value of **Binder's fourth-order cumulant** $U_4 = 1 - \frac{\langle M^4 \rangle}{3\langle M^2 \rangle^2}$ in this ordered thermodynamic limit? *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the ratio of the fourth moment to the square of the second moment when the distribution is concentrated entirely at fixed spontaneous magnetization magnitudes.

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q23_34')

**Q34 · medium** · 💡

At a certain temperature $T$ in a Monte Carlo simulation of a $20 \times 20$ Ising lattice, you measure the second and fourth magnetization moments to be $\langle M^2 \rangle = 0.250$ and $\langle M^4 \rangle = 0.075$. Calculate Binder's 4th-order cumulant $U_4(T, L)$ at this state point. *(Round to 2 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the definition of the fourth-order Binder cumulant using the measured magnetization moments.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q23_35')

**Q35 · medium** · 💡

What is the primary practical advantage of using the **Binder cumulant crossing** $U_4(T, L)$ to locate the critical temperature $T_c$, compared to performing a full data collapse of $M(T, L)\\,L^{\beta/\nu}$ vs. $t L^{1/\nu}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check whether $U_4(T, L)$ has any $L$-dependent prefactor that requires knowing $\beta$ or $\nu$ in advance.

</div>
</details>

- [ ] **(A)** Because the $L^{-4\beta/\nu}$ scaling prefactors cancel out between $\langle M^4 \rangle$ and $\langle M^2 \rangle^2$, $U_4(T, L)$ curves for different $L$ intersect directly at $T_c$ without needing any prior knowledge or simultaneous fitting of $\beta$ or $\nu$.
- [ ] **(B)** Binder's cumulant requires running at only a single lattice size $L$ and a single temperature $T$.
- [ ] **(C)** Binder's cumulant eliminates critical slowing down so local Metropolis has $z = 0$.
- [ ] **(D)** Binder's cumulant is identically equal to $T_c$ at all temperatures.

In [46]:
ws.render_section('q23_36')

**Q36 · hard** · 💡

The temperature derivative (slope) of Binder's cumulant at the critical crossing point varies systematically with lattice size $L$. In a 3D Ising simulation, you measure this slope magnitude at $T_c$ to be $S(L=10) = 1.80$ for $L=10$ and $S(L=20) = 5.41$ for $L=20$. What value of the correlation-length critical exponent $\nu$ do these measurements yield? *(Round to 2 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the dimensionless scaling argument of the Binder cumulant with respect to temperature and observe how the resulting prefactor scales with system size.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q23_37')

**Q37 · medium** · 💡

How does the shape of the finite-size magnetization probability distribution $P_L(M)$ (at $h = 0$) evolve as a system is cooled from $T > T_c$, through $T = T_c$, to $T < T_c$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the transition from central-limit fluctuations around zero, through scale-invariant critical fluctuations, to broken-symmetry bimodal ordering.

</div>
</details>

- [ ] **(A)** From a single narrow Gaussian peak centered at $M = 0$ ($T > T_c$), to a broad non-Gaussian critical distribution of width $\propto L^{-\beta/\nu}$ ($T = T_c$), to a symmetric bimodal distribution with two sharp peaks at $\pm M_{\text{spont}}$ ($T < T_c$).
- [ ] **(B)** From a bimodal distribution at $T > T_c$ to a single delta function at $M = 0$ at $T < T_c$.
- [ ] **(C)** It remains a pure Gaussian centered at $M = 0$ at all temperatures, with only its variance changing.
- [ ] **(D)** It is uniform on $[-1, 1]$ for all $T > T_c$.

In [48]:
ws.render_section('q23_38')

**Q38 · hard** · 💡

At the critical point $T = T_c$ of a $d = 3$ Ising model with anomalous dimension $\eta = 0.032$, the spin–spin correlation function decays algebraically with distance as $G(r) \propto r^{-p}$. What is the numerical value of the spatial power-law decay exponent $p$? *(Round to 3 decimal places.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the definition of the Fisher anomalous dimension $\eta$ in the critical two-point correlation function for a $d$-dimensional system.

</div>
</details>

_Numeric answer required._

In [49]:
ws.render_section('q23_39')

**Q39 · medium** · 💡

In a zero-field ($h = 0$) Monte Carlo simulation of a finite Ising lattice below $T_c$, why do practitioners measure $\langle |M| \rangle$ (and define the finite-size susceptibility via $\chi' \propto L^d(\langle M^2 \rangle - \langle |M| \rangle^2)$) rather than using the signed average $\langle M \rangle$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the symmetry $H(\\{-s_i\\}) = H(\\{s_i\\})$ of the zero-field Ising Hamiltonian and the bimodal peaks at $\pm M_0$.

</div>
</details>

- [ ] **(A)** Because at $h = 0$, the Hamiltonian is invariant under global spin inversion ($s_i \to -s_i$), so $P_L(M) = P_L(-M)$ is strictly symmetric and the true ensemble average $\langle M \rangle$ is identically zero for any finite $L$.
- [ ] **(B)** Because $M$ is a complex number below $T_c$.
- [ ] **(C)** Because $|M|$ is an analytic function in the thermodynamic limit.
- [ ] **(D)** Because $\langle M^2 \rangle$ is always negative below $T_c$.

In [50]:
ws.render_section('q23_40')

**Q40 · hard** · 💡

Consider the pseudocritical coupling $K_c(L) = J / (k_B T_c(L))$ determined from five different observables ($M, \chi, \ln|M|, \ln M^2, U_L$) plotted against $L^{-1/\nu}$ for the 3D Ising model. Why do all five curves converge to the exact same vertical intercept $K_c(\infty) \approx 0.22165$ as $L^{-1/\nu} \to 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider how finite-size shifts in observable extrema depend on box size and how they behave in the thermodynamic limit.

</div>
</details>

- [ ] **(A)** Each finite-size observable peaks or crosses at its own scaling coordinate $x_{\text{obs}}$, giving a linear shift $K_c(L) = K_c(\infty) + a_{\text{obs}}\\,L^{-1/\nu}$ with observable-dependent slope $a_{\text{obs}}$, but in the thermodynamic limit ($L^{-1/\nu} \to 0$) all observables must converge to the unique bulk critical point $K_c(\infty)$.
- [ ] **(B)** All five observables have the exact same slope $a_{\text{obs}} = 0$ for all $L$.
- [ ] **(C)** The horizontal axis $L^{-1/\nu}$ diverges to $\infty$ as $L \to \infty$.
- [ ] **(D)** Because $K_c(\infty)$ is a universal constant equal to $0.22165$ for all lattices in all dimensions.

In [51]:
ws.render_section('q23_41')

**Q41 · hard** · 💡

Combining the four fundamental static scaling laws (Rushbrooke, Widom, Fisher, and Josephson hyperscaling), express the critical isotherm exponent $\delta$ as a function of the spatial dimensionality $d$ and the anomalous dimension $\eta$. Evaluate the resulting value of $\delta$ for the 2D Ising model where $d = 2$ and $\eta = 1/4 = 0.25$. *(Enter an exact integer.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Eliminate $\alpha, \beta, \gamma,$ and $\nu$ from the scaling relations to express $\delta$ in terms of $d$ and $\eta$, then evaluate for the 2D critical parameters.

</div>
</details>

_Numeric answer required._

In [52]:
ws.render_section('q23_42')

**Q42 · medium** · 💡

Which of the following statements represent valid synthesis principles for simulating continuous phase transitions in finite boxes? *(Select all that apply.)*

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review cluster updates, RG fixed points, FSS data collapse, and Binder's cumulant.

</div>
</details>

- [ ] **(A)** Swendsen–Wang and Wolff cluster algorithms leave static equilibrium averages ($M, \chi, C_V, U_4$) unchanged while drastically reducing the dynamical exponent $z$ near $T_c$.
- [ ] **(B)** Under Kadanoff–Wilson RG coarse-graining, the critical point $T = T_c$ is a non-trivial unstable fixed point ($K^* = R(K^*)$) where $\xi = \infty$.
- [ ] **(C)** Plotting $M(T, L)\\,L^{\beta/\nu}$ against $t L^{1/\nu}$ collapses data from multiple box sizes $L$ onto a universal scaling curve.
- [ ] **(D)** To extract bulk critical exponents accurately, one should simulate only a single box size $L$ far away from $T_c$ and ignore finite-size effects.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()